In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.silver.returns")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.silver.returns (
    return_id STRING,
    transaction_id STRING,
    return_date DATE,
    reason STRING
)
""")

In [0]:
spark.sql("""
INSERT INTO de_mini_project.silver.returns (return_id, transaction_id, return_date, reason)
WITH returns_cleaned AS (
    SELECT 
        rtn_id_no AS return_id,
        orgnl_trxn_id_ AS transaction_id,
        COALESCE(
            try_to_date(rtn_date_string, 'dd-MM-yyyy'),
            try_to_date(rtn_date_string, 'yyyy.MM.dd'),
            try_to_date(rtn_date_string, 'dd-MMM-yy'),
            try_to_date(rtn_date_string, 'MM/dd/yyyy'),
            try_to_date(rtn_date_string, 'MMM dd, yyyy')
        ) AS return_date,
        rsn_code_ AS reason
    FROM de_mini_project.azure_blob_storage.transactions
)
SELECT 
    return_id, 
    transaction_id, 
    return_date, 
    reason
FROM returns_cleaned
QUALIFY ROW_NUMBER() OVER (PARTITION BY return_id ORDER BY return_date DESC) = 1
""")

In [0]:
display(spark.sql("SELECT * FROM de_mini_project.silver.returns"))